# 05 — Joint weather + hazard training with adaptive loss balancing

The ClimaX encoder is **fine-tuned** for five tasks at once: the 72 h weather forecast, the hypsometric physics loss,
and the heatwave, extreme-wind and extreme-rainfall heads. Instead of a hand-set λ, each task gets a **learned weight**
via uncertainty weighting (Kendall, Gal & Cipolla, CVPR 2018): `L = Σ exp(−s_i)·L_i + s_i`.

Steps:
1. **Frozen baseline on the corrected 6×6 grid.** Hazard heads on the frozen λ=0 and λ=0.001 encoders, plus climatology,
   persistence and logistic regression (same as notebook 04, but on the corrected grid).
2. **Joint training.** Starts from the λ=0 checkpoint (`INIT`), fine-tunes everything with learned task weights, and selects
   the epoch with the best mean validation PR-AUC.
3. **Comparison.** Joint vs frozen on identical test samples, with paired 7-day block-bootstrap CIs, plus weather RMSE vs
   persistence and climatology.

**Attach:** the corrected 6×6 dataset, `checkpoints-for-regional-forecasting` (λ=0.001), your λ=0 checkpoint dataset, and the IMD rainfall.
**Accelerator:** GPU (one T4 is enough). Keep `SMOKE_TEST = True` first (~10 min) to check the pipeline, then set it to `False`
and use *Save Version → Save & Run All (Commit)*.

In [ ]:
SMOKE_TEST = True              # True: tiny run to check everything; False: real experiment
INIT = "nophysics"             # encoder to start from: "nophysics" (λ=0), "physics" (λ=0.001) or "scratch"
BALANCING = "uncertainty"      # "uncertainty" (adaptive) or "fixed" (internship weights, for ablation)
EPOCHS = 15
SAMPLES_PER_EPOCH = 65536      # random hourly pairs per epoch (of ~280k)
BATCH_SIZE = 128
RUN_FROZEN_BASELINE = True     # step 1 (needed for the comparison; reused if its output already exists)

INPUT_ROOT = "/kaggle/input"
CORRECTED_DIR_OVERRIDE = None
IMD_DIR_OVERRIDE = None
PHYSICS_CKPT_OVERRIDE = None
NOPHYSICS_CKPT_OVERRIDE = None
REPO_URL = "https://github.com/Aditya1Agrawal/ClimaX-Monsoon-PINN.git"
REPO_BRANCH = "mtech-extreme-events"
WORK = "/kaggle/working"

In [ ]:
import os, sys, json, subprocess, pathlib, importlib.util, time
REPO_DIR = f"{WORK}/ClimaX-Monsoon-PINN"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
print("repo commit:", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.6.12"], check=True)
timm_init = pathlib.Path(importlib.util.find_spec("timm").origin).parent / "models" / "__init__.py"
txt = timm_init.read_text()
if sys.version_info >= (3, 11) and "\nfrom .maxxvit import *" in txt:
    timm_init.write_text(txt.replace("\nfrom .maxxvit import *", "\n# from .maxxvit import *  # disabled for py>=3.11"))
sys.path.insert(0, f"{REPO_DIR}/src")
import torch, numpy as np, pandas as pd
print("torch", torch.__version__, "| GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## Inputs

In [ ]:
from climax_hazards import inspect_utils as iu
cands = [p for p in iu.find_dataset_dirs(INPUT_ROOT)
         if np.load(p / "lon.npy").size == 6 and not iu.detect_lon_padding(p)["looks_padded"]]
CORR_DIR = str(iu.resolve_single(cands, CORRECTED_DIR_OVERRIDE, "corrected 6x6 dataset"))
print("corrected dataset:", CORR_DIR, "| lon:", np.load(f"{CORR_DIR}/lon.npy").tolist())

def find_imd(root):
    for dirpath, _, files in os.walk(root):
        if "1979.grd" in files:
            return dirpath
IMD_DIR = IMD_DIR_OVERRIDE or find_imd(INPUT_ROOT)
print("IMD:", IMD_DIR or "NOT FOUND - extreme rainfall will be skipped")

rows = []
for p in iu.find_checkpoints(INPUT_ROOT):
    ck = iu.load_checkpoint(p)
    mc = next((v for k, v in (ck.get("callbacks") or {}).items() if k.startswith("ModelCheckpoint")), {})
    rows.append({"path": str(p), "run_dir": str(p.parent), "file": p.name,
                 "lambda": ck.get("hyper_parameters", {}).get("physics_lambda"), "epoch": ck.get("epoch"),
                 "best_at_save": os.path.basename(mc.get("best_model_path", "") or "")})
    del ck
ckpts = pd.DataFrame(rows)
final = ckpts.sort_values("epoch").groupby("run_dir").tail(1).set_index("run_dir")
ckpts["run_final_best"] = [r.file == final.loc[r.run_dir, "best_at_save"] for r in ckpts.itertuples()]
print(ckpts.drop(columns=["path", "run_dir"]).to_string(index=False))

def pick(lam, override):
    if override:
        return os.path.abspath(override)
    sel = ckpts[(ckpts["lambda"] == lam) & ckpts["run_final_best"]]
    if len(sel) != 1:
        raise RuntimeError(f"expected one final-best checkpoint with lambda={lam}, found {len(sel)}; set the override")
    return os.path.abspath(sel["path"].iloc[0])
PHYS = pick(0.001, PHYSICS_CKPT_OVERRIDE)
NOPHYS = pick(0.0, NOPHYSICS_CKPT_OVERRIDE)
print("\nphysics (λ=0.001):", PHYS, "\nno physics (λ=0):", NOPHYS)

In [ ]:
WORK = os.path.abspath(WORK)   # scripts run from the repo folder, so all output paths must be absolute
env = dict(os.environ, PYTHONPATH=f"{REPO_DIR}/src")
def run(cmd, log_path):
    t0 = time.time()
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            print(line, end="", flush=True)
        rc = proc.wait()
    print(f"--> exit {rc} after {(time.time() - t0) / 60:.1f} min")
    assert rc == 0, f"failed, see {log_path}"
tag = "_smoke" if SMOKE_TEST else ""

## 1. Frozen-encoder baseline on the corrected grid

In [ ]:
FROZEN_DIR = f"{WORK}/frozen_corrected{tag}"
if RUN_FROZEN_BASELINE and not os.path.exists(f"{FROZEN_DIR}/hazard_summary_test.csv"):
    os.makedirs(FROZEN_DIR, exist_ok=True)
    cmd = [sys.executable, "scripts/mtech/run_hazard_experiment.py", "--data_dir", CORR_DIR, "--out_dir", FROZEN_DIR,
           "--checkpoint", f"frozen_nophysics={NOPHYS}", "--checkpoint", f"frozen_physics={PHYS}",
           "--compare", "frozen_nophysics:frozen_physics", "--compare", "frozen_nophysics:logistic", "--compare", "frozen_physics:logistic"]
    if IMD_DIR:
        cmd += ["--imd_dir", IMD_DIR]
    if SMOKE_TEST:
        cmd += ["--max_samples", "300", "--epochs", "2", "--seeds", "0", "--no_logistic"]
    run(cmd, f"{FROZEN_DIR}/run.log")
print(pd.read_csv(f"{FROZEN_DIR}/hazard_summary_test.csv").round(4).to_string(index=False))

## 2. Joint training with adaptive loss balancing

In [ ]:
JOINT_DIR = f"{WORK}/joint_{BALANCING}_init-{INIT}{tag}"
os.makedirs(JOINT_DIR, exist_ok=True)
srcs = [s for s in ("frozen_nophysics", "frozen_physics", "logistic", "climatology", "persistence")
        if os.path.exists(f"{FROZEN_DIR}/preds/{s}.npz")]
cmd = [sys.executable, "scripts/mtech/train_joint.py", "--data_dir", CORR_DIR, "--out_dir", JOINT_DIR,
       "--balancing", BALANCING, "--epochs", str(EPOCHS), "--samples_per_epoch", str(SAMPLES_PER_EPOCH),
       "--batch_size", str(BATCH_SIZE), "--compare_dir", FROZEN_DIR, "--compare_sources", *srcs]
cmd += ["--from_scratch"] if INIT == "scratch" else ["--init_ckpt", NOPHYS if INIT == "nophysics" else PHYS]
if IMD_DIR:
    cmd += ["--imd_dir", IMD_DIR]
if SMOKE_TEST:
    cmd += ["--smoke"]
run(cmd, f"{JOINT_DIR}/run.log")

## 3. Results

In [ ]:
hist = pd.read_csv(f"{JOINT_DIR}/history.csv")
print("Learned task weights and validation scores per epoch:")
cols = ["epoch"] + [c for c in hist.columns if c.startswith("weight_")] + ["val_w_mse", "val_rmse_geopotential_500"] + [c for c in hist.columns if c.startswith("val_ap")]
print(hist[cols].round(4).to_string(index=False))

joint = pd.read_csv(f"{JOINT_DIR}/hazard_results.csv")
joint = joint[joint.split == "test"][["source", "hazard", "pr_auc", "roc_auc", "f1", "precision", "recall", "prevalence"]]
frozen = pd.read_csv(f"{FROZEN_DIR}/hazard_summary_test.csv")[["source", "hazard", "pr_auc", "roc_auc", "f1", "precision", "recall", "prevalence"]]
both = pd.concat([frozen, joint]).sort_values(["hazard", "pr_auc"])
print("\nTEST hazards (PR-AUC is primary; no-skill PR-AUC = prevalence):")
print(both.round(4).to_string(index=False))
if os.path.exists(f"{JOINT_DIR}/bootstrap_vs_frozen.csv"):
    print("\nPaired 7-day block bootstrap, PR-AUC(joint) - PR-AUC(b), 95% CI:")
    print(pd.read_csv(f"{JOINT_DIR}/bootstrap_vs_frozen.csv").round(4).to_string(index=False))

wt = pd.read_csv(f"{JOINT_DIR}/weather_test.csv")
print("\nTEST weather RMSE (joint model vs baselines; hyps_resid_* = physical-consistency error in metres):")
print(wt.pivot_table(index="variable", columns="source", values="rmse", sort=False).round(3).to_string())
print(json.dumps(json.load(open(f"{JOINT_DIR}/run_meta.json")), indent=1)[:1500])

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c in [c for c in hist.columns if c.startswith("weight_")]:
    ax[0].plot(hist["epoch"], hist[c], marker="o", label=c.replace("weight_", ""))
ax[0].set_yscale("log"); ax[0].set_title("learned task weights exp(-s)"); ax[0].set_xlabel("epoch"); ax[0].legend()
for c in [c for c in hist.columns if c.startswith("val_ap_") and c != "val_ap_mean"]:
    ax[1].plot(hist["epoch"], hist[c], marker="o", label=c.replace("val_ap_", ""))
ax[1].set_title("validation PR-AUC"); ax[1].set_xlabel("epoch"); ax[1].legend()
fig.tight_layout(); fig.savefig(f"{JOINT_DIR}/training_curves.png", dpi=120); plt.show()
print("outputs:", sorted(os.listdir(JOINT_DIR)))